In [ ]:
!pip install ucimlrepo

In [ ]:
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import Normalizer
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
spam_data = fetch_ucirepo(id=94)

In [ ]:
spam_df = spam_data.data.original
spam_df

In [ ]:
spam_features = spam_df.iloc[:, :-1]
spam_target   = spam_df.iloc[:, -1]

In [ ]:
spam_df.describe()

In [ ]:
se = Normalizer()
spam_features = pd.DataFrame(
    se.fit_transform(spam_features),
    columns=spam_df.columns[:-1]
)

In [ ]:
pd.concat([spam_features, spam_target], axis=1)

In [ ]:
corr = spam_df.corr()
target_corr = corr['Class'].sort_values(ascending=False)
print(target_corr)

## Manual Logistic Regression via Gradient Descent

Logistic regression models the probability that a sample belongs to class 1
by applying the **sigmoid function** to the linear combination of features:

$$\hat{p}^{(i)} = \sigma(z^{(i)}) = \frac{1}{1 + e^{-z^{(i)}}}, \quad z^{(i)} = \mathbf{w}^\top \mathbf{x}^{(i)} + b$$

The model is trained by minimising the **Binary Cross-Entropy** (log-loss) cost:

$$J(\mathbf{w}) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log \hat{p}^{(i)} + (1 - y^{(i)}) \log (1 - \hat{p}^{(i)}) \right]$$

The **vectorised** gradient descent update rules are:

$$\frac{\partial J}{\partial \mathbf{w}} = \frac{1}{m} X^\top (\hat{\mathbf{p}} - \mathbf{y}), \quad \frac{\partial J}{\partial b} = \frac{1}{m} \mathbf{1}^\top(\hat{\mathbf{p}} - \mathbf{y})$$

$$\mathbf{w} \leftarrow \mathbf{w} - \alpha \cdot \frac{\partial J}{\partial \mathbf{w}}, \quad b \leftarrow b - \alpha \cdot \frac{\partial J}{\partial b}$$

> **Why NumPy?** The loop-based version iterated over every sample **and** every feature in pure Python — O(m x d) overhead per epoch. Replacing this with `X @ w` (a single BLAS matrix-multiply) gives a **~100-500x speedup** on this 4601 x 57 dataset.

In [ ]:
# Convert to NumPy arrays once — all math runs in C/BLAS after this
X = spam_features.to_numpy()               # shape (4601, 57)
y = spam_target.to_numpy().astype(float)   # shape (4601,)

n, feat_count = X.shape                    # 4601 samples, 57 features

w      = np.zeros(feat_count)              # shape (57,)
b      = 0.0
ALPHA  = 0.1
EPOCHS = 1000


def sigmoid(z: np.ndarray) -> np.ndarray:
    """Element-wise sigmoid with clipping to prevent overflow."""
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))


def predict_prob(w: np.ndarray, b: float) -> np.ndarray:
    """
    Compute P(y=1 | X) for all n samples simultaneously.
    X @ w is a single matrix multiply (BLAS) replacing the old double for-loop.
    Returns shape (n,).
    """
    return sigmoid(X @ w + b)


def cost(w: np.ndarray, b: float) -> float:
    """
    Binary Cross-Entropy loss (fully vectorised):
      J = -(1/m) * [ y . log(p) + (1-y) . log(1-p) ]
    np.mean replaces the Python sum loop.
    """
    p = np.clip(predict_prob(w, b), 1e-15, 1 - 1e-15)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))


def new_weights(w: np.ndarray, b: float):
    """
    One vectorised gradient descent step.

    residual = p_hat - y                  shape (n,)
    dw       = (1/m) * X^T @ residual     shape (57,)  -- one BLAS call
    db       = mean(residual)             scalar
    """
    residual = predict_prob(w, b) - y      # (n,)
    dw = (X.T @ residual) / n              # (57,)
    db = np.mean(residual)
    return w - ALPHA * dw, b - ALPHA * db

## Training Loop

In [ ]:
cost_history = []

for epoch in range(EPOCHS):
    w, b = new_weights(w, b)
    if epoch % 100 == 0:
        c = cost(w, b)
        cost_history.append((epoch, c))
        print(f"Epoch {epoch:>4}  |  Log-loss: {c:.6f}")

print(f"\nTraining complete. Final log-loss: {cost(w, b):.6f}")

## Evaluation

In [ ]:
THRESHOLD = 0.5

probs       = predict_prob(w, b)                   # shape (n,)
predictions = (probs >= THRESHOLD).astype(int)
actuals     = y.astype(int)

TP = int(np.sum((predictions == 1) & (actuals == 1)))
TN = int(np.sum((predictions == 0) & (actuals == 0)))
FP = int(np.sum((predictions == 1) & (actuals == 0)))
FN = int(np.sum((predictions == 0) & (actuals == 1)))

accuracy  = (TP + TN) / n
precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"\nConfusion Matrix:")
print(f"  TP={TP}  FP={FP}")
print(f"  FN={FN}  TN={TN}")

## Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# --- 1: Cost curve ---
epochs_logged, costs_logged = zip(*cost_history)
axes[0].plot(epochs_logged, costs_logged, marker='o', markersize=4)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Cross-Entropy')
axes[0].set_title('Log-Loss vs Epoch')

# --- 2: Confusion matrix heatmap ---
cm = np.array([[TP, FP], [FN, TN]])
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
    xticklabels=['Pred Spam', 'Pred Ham'],
    yticklabels=['Actual Spam', 'Actual Ham']
)
axes[1].set_title('Confusion Matrix')

# --- 3: Predicted probability distribution by class ---
spam_probs = probs[actuals == 1]
ham_probs  = probs[actuals == 0]
axes[2].hist(ham_probs,  bins=40, alpha=0.6, label='Ham (0)',  color='steelblue')
axes[2].hist(spam_probs, bins=40, alpha=0.6, label='Spam (1)', color='tomato')
axes[2].axvline(THRESHOLD, color='black', linestyle='--', linewidth=1.2, label=f'Threshold={THRESHOLD}')
axes[2].set_xlabel('Predicted Probability')
axes[2].set_ylabel('Count')
axes[2].set_title('Score Distribution by Class')
axes[2].legend()

plt.tight_layout()
plt.show()